# Biomechanics of Movement (BME 4903): Jump Landing Analysis
## OpenSim Inverse Kinetics & Knee Extensor Muscle Forces across Landing Postures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LTUebl/jump-landing-opensim/blob/main/JumpLanding_OpenSim_Analysis.ipynb)
[![Interactive Web Guide](https://img.shields.io/badge/Public_Site-Interactive_Dashboard-blue?logo=googlechrome&logoColor=white)](https://ltuebl.github.io/jump-landing-opensim/)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repository-181717?logo=github&logoColor=white)](https://github.com/LTUebl/jump-landing-opensim)

**Instructor / Lab Guide**  
*Course:* BME 4903 Advanced Kinesiology / Biomechanics of Movement  
*Affiliation:* Lawrence Technological University · Dept. of Biomedical Engineering  
*Topic:* Single-Cycle Jump Landing: Impact Attenuation & Musculoskeletal Demand  

---

### Objectives:
1. **Automate Kinematic Event Detection:** Automatically identify key landing events (**Jump Apex**, **Initial Contact [IC]**, and **Maximum Knee Flexion [KM]**) from OpenCap kinematic motion capture data without requiring manual multi-cycle segmentation.
2. **Inverse Kinetics (Ground Reaction Forces):** Calculate total vertical Ground Reaction Force ($vGRF$) in Newtons and normalized to Body Weight ($BW$). Determine peak forces, loading rates, and vertical impulse.
3. **Knee Extensor Muscle Forces:** Quantify muscle forces for the primary knee extensor group (Quadriceps Femoris):
   - **Vastus Lateralis (`vaslat`)**
   - **Vastus Medialis (`vasmed`)**
   - **Vastus Intermedius (`vasint`)**
   - **Rectus Femoris (`recfem`, biarticular)**
4. **Posture Comparison:** Compare biomechanical differences between **Forward Trunk Lean** vs. **Upright Trunk** landing postures.
5. **Clinical & Injury Implications:** Evaluate the relationship between trunk posture, quadriceps forces, knee flexion angles, and Anterior Cruciate Ligament (ACL) injury risk.


---
## Part 0: Environment Setup (Fast & Crash-Free)

This notebook is configured to run smoothly in Google Colab without restarting the kernel or crashing:
* All core data analysis and plotting packages (`numpy`, `pandas`, `scipy`, `matplotlib`) are pre-installed in Google Colab.
* We install `openpyxl` (for Excel export) and `pyopensim` directly via `pip`.
* **No kernel restarts are required!** You can click **"Run All"** and the entire notebook will execute from top to bottom in seconds.


In [ ]:
# Cell 0: Environment Setup (No kernel restart required!)
import os
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

# Install openpyxl for Excel report generation and pyopensim if available
!pip install -q openpyxl pyopensim

# Import required analysis libraries
import sys, glob, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# Test OpenSim availability
try:
    import opensim
    print(f"✓ OpenSim package active! Version: {opensim.GetVersion()}")
except ImportError:
    print("✓ Running in native high-speed Python mode (built-in OpenSim storage parser active).")

print("✓ All libraries loaded! Ready for Part 1.")


---
## Part 1: Load OpenCap Jump Landing Session Data

You can analyze either:
- **Option A (Student Data):** Upload your own OpenCap session `.zip` file downloaded from the OpenCap web portal.
- **Option B (Example Dataset):** Use the pre-loaded Subject EGM dataset (Subject mass = 95 kg, height = 1.95 m) containing `LeaningJump` and `UprightJump`.

### How to upload your OpenCap `.zip` file in Google Colab:
There are **two easy ways** to upload:
1. **Method 1 (Interactive Upload Button):** Run the first cell below and click the **`Choose Files`** button that appears. Select your OpenCap `.zip` file from your computer.
2. **Method 2 (Sidebar Drag-and-Drop):** 
   - Click the **Folder icon (📁)** on the left sidebar of Google Colab.
   - Drag and drop your `.zip` file from your computer directly into the file list (or click the **Upload to session storage** button with the upward arrow).
   - Wait for the circular upload ring at the bottom of the sidebar to finish (100%).


In [ ]:
# Part 1.1: Interactive File Upload Widget (Google Colab)
# Run this cell to upload your OpenCap session .zip file directly.
# If you want to use the pre-loaded Subject EGM example data instead, simply cancel or skip uploading.

import os, glob, shutil, zipfile

try:
    from google.colab import files
    print("Click 'Choose Files' below to upload your OpenCap session .zip file:")
    uploaded = files.upload()
    if uploaded:
        for fn in uploaded.keys():
            print(f"✓ Successfully uploaded: {fn} ({len(uploaded[fn]) / (1024*1024):.2f} MB)")
    else:
        print("No file uploaded via button. Checking for existing files or example data...")
except ImportError:
    print("Running in local Python environment (not Google Colab). Checking local directory...")


In [ ]:
# Part 1.2: Extract Session Archive & Set Subject Parameters
# Set your subject's body mass in kilograms (Default: 95.0 kg for Subject EGM)
SUBJECT_MASS_KG = 95.0 

session_dir = None

# 1. Check for any uploaded or existing .zip files
zip_candidates = glob.glob("*.zip") + glob.glob("/content/*.zip") + glob.glob("../*.zip")

# Auto-download example data from GitHub if no files exist
has_local_data = any(os.path.isdir(p) for p in [
    "OpenCapData_aacdc355-04b6-4d74-97e8-0dcd8d28a628", 
    "JumpLandingOpenCapExampleData_EGM",
    "/content/OpenCapData_aacdc355-04b6-4d74-97e8-0dcd8d28a628"
])
if not zip_candidates and not has_local_data:
    print("No local zip detected. Downloading Subject EGM example dataset from GitHub...")
    import urllib.request
    egm_url = "https://raw.githubusercontent.com/LTUebl/jump-landing-opensim/main/data/JumpLanding_OpenCap_OpenSimData_EGM.zip"
    try:
        target_save = "/content/JumpLanding_OpenCap_OpenSimData_EGM.zip" if os.path.exists("/content") else "JumpLanding_OpenCap_OpenSimData_EGM.zip"
        urllib.request.urlretrieve(egm_url, target_save)
        print(f"✓ Downloaded example dataset: {target_save} (1.39 MB)")
        zip_candidates = [target_save]
    except Exception as e:
        print(f"Note: Could not auto-download example ({e}). Please upload your OpenCap session .zip.")

# Prioritize custom student uploaded zips first
custom_zips = [z for z in zip_candidates if "OpenSimData_EGM" not in z and "ExampleData_EGM" not in z]
target_zip = custom_zips[0] if custom_zips else (zip_candidates[0] if zip_candidates else None)

if target_zip:
    extract_target = "/content" if os.path.exists("/content") else "."
    print(f"Extracting session archive: {target_zip} into {extract_target}...")
    with zipfile.ZipFile(target_zip, 'r') as zf:
        zf.extractall(extract_target)

# 2. Search for the directory containing 'OpenSimData'
search_paths = ["/content", ".", "..", "JumpLandingOpenCapExampleData_EGM", "../JumpLandingOpenCapExampleData_EGM"]
for spath in search_paths:
    if os.path.exists(spath):
        for root, dirs, files_in_dir in os.walk(spath):
            if "OpenSimData" in dirs:
                kin_dir = os.path.join(root, "OpenSimData", "Kinematics")
                if os.path.isdir(kin_dir) and glob.glob(os.path.join(kin_dir, "*.mot")):
                    session_dir = os.path.abspath(root)
                    break
        if session_dir:
            break

if session_dir is None:
    raise FileNotFoundError(
        "Could not find 'OpenSimData' directory! Please upload your OpenCap session .zip file "
        "either by clicking 'Choose Files' in the cell above or dragging it into the left sidebar (Folder icon 📁)."
    )

print(f"✓ Active Session Directory: {session_dir}")
kin_files = glob.glob(os.path.join(session_dir, "OpenSimData", "Kinematics", "*.mot"))
kin_files = [f for f in kin_files if not os.path.basename(f).startswith(".")]
print(f"✓ Found {len(kin_files)} kinematic trials: {[os.path.basename(f) for f in kin_files]}")


---
## Part 2: Step 1 - Single-Cycle Kinematic Event Detection
In jump landing protocols, each trial represents a single jump-landing cycle.
Unlike cyclic walking or running where multi-cycle cropping (`step5a_crop_mot.py`) was previously needed, we automatically identify the single landing cycle using vertical kinematics:
- **Jump Apex ($t_{apex}$):** The maximum jump height (maximum pelvis vertical position / minimum vertical velocity).
- **Initial Contact ($t_{IC}$):** The instant of ground contact (rapid deceleration of foot/heel or rapid onset of knee flexion).
- **Knee Maximum Flexion ($t_{KM}$):** The bottom of the landing absorption phase (peak knee flexion angle).
- **Absorption Phase Duration:** $\Delta t_{abs} = t_{KM} - t_{IC}$.


In [ ]:
# Storage File Reader Utility
def read_storage(path):
    with open(path, "r") as f:
        for idx, line in enumerate(f):
            if line.strip().lower() == "endheader":
                df = pd.read_csv(path, sep=r"\s+", skiprows=idx + 1)
                df.columns = [c.strip() for c in df.columns]
                return df
    raise ValueError(f"Could not parse storage file: {path}")

def detect_landing_events(df_kin):
    time = df_kin["time"].values
    dt = np.median(np.diff(time))

    # Pelvis Y or Knee Angle
    pelvis_y = df_kin["pelvis_ty"].values if "pelvis_ty" in df_kin.columns else None
    knee_col = "knee_angle_r" if "knee_angle_r" in df_kin.columns else "knee_flexion_r"
    knee_angle = np.abs(df_kin[knee_col].values) if knee_col in df_kin.columns else None

    # 1. Apex
    if pelvis_y is not None:
        idx_apex = np.argmax(pelvis_y)
    else:
        idx_apex = len(time) // 3
    t_apex = time[idx_apex]

    # 2. Knee Max (KM)
    post_apex = np.where(time > t_apex)[0]
    if len(post_apex) > 0 and knee_angle is not None:
        idx_km = post_apex[np.argmax(knee_angle[post_apex])]
    else:
        idx_km = len(time) - 1
    t_km = time[idx_km]

    # 3. Initial Contact (IC)
    idx_ic = None
    if knee_angle is not None:
        window = np.arange(idx_apex, idx_km)
        if len(window) > 4:
            d_knee = np.gradient(knee_angle[window], dt)
            onset = np.where(d_knee > 40.0)[0]
            if len(onset) > 0:
                idx_ic = window[onset[0]]

    if idx_ic is None:
        idx_ic = int(idx_apex + 0.35 * (idx_km - idx_apex))
    t_ic = time[idx_ic]

    # Posture at KM (Lumbar Extension angle)
    lumbar_col = "lumbar_extension" if "lumbar_extension" in df_kin.columns else None
    lumbar_km = df_kin[lumbar_col].values[idx_km] if lumbar_col else 0.0
    posture_desc = "Forward Lean" if lumbar_km < -25.0 else "Upright"

    return {
        "t_apex": t_apex,
        "t_ic": t_ic,
        "t_km": t_km,
        "duration": t_km - t_ic,
        "peak_knee": knee_angle[idx_km] if knee_angle is not None else 0.0,
        "lumbar_km": lumbar_km,
        "posture": posture_desc
    }


In [ ]:
# Run Event Detection on Kinematics
kin_dir = os.path.join(session_dir, "OpenSimData", "Kinematics")
kin_files = glob.glob(os.path.join(kin_dir, "*.mot"))
kin_files = [f for f in kin_files if not os.path.basename(f).startswith(".")]

event_results = []
fig, axs = plt.subplots(len(kin_files), 1, figsize=(12, 4.5 * len(kin_files)), sharex=False)
if len(kin_files) == 1: axs = [axs]

for idx, fpath in enumerate(kin_files):
    trial_name = os.path.splitext(os.path.basename(fpath))[0]
    df_kin = read_storage(fpath)
    ev = detect_landing_events(df_kin)
    ev["trial"] = trial_name
    event_results.append(ev)

    # Plot
    ax = axs[idx]
    time = df_kin["time"].values
    knee_col = "knee_angle_r" if "knee_angle_r" in df_kin.columns else "knee_flexion_r"
    knee_deg = np.abs(df_kin[knee_col].values) if knee_col in df_kin.columns else np.zeros_like(time)
    
    ax.plot(time, knee_deg, color="#1f77b4", lw=2, label="Knee Flexion (°)")
    if "pelvis_ty" in df_kin.columns:
        ax.plot(time, df_kin["pelvis_ty"] * 50, color="#2ca02c", linestyle="--", alpha=0.7, label="Pelvis Height (x50 cm)")

    # Event lines
    ax.axvline(ev["t_apex"], color="purple", linestyle=":", lw=2, label=f"Apex ({ev['t_apex']:.2f} s)")
    ax.axvline(ev["t_ic"], color="red", linestyle="--", lw=2, label=f"Initial Contact ({ev['t_ic']:.2f} s)")
    ax.axvline(ev["t_km"], color="darkorange", linestyle="-.", lw=2, label=f"Knee Max ({ev['t_km']:.2f} s)")
    
    # Shade absorption phase
    ax.axvspan(ev["t_ic"], ev["t_km"], color="gold", alpha=0.2, label=f"Absorption ({ev['duration']:.2f} s)")
    ax.set_title(f"Trial: {trial_name} — Posture: {ev['posture']} (Lumbar = {ev['lumbar_km']:.1f}°, Peak Knee = {ev['peak_knee']:.1f}°)", fontweight="bold")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Angle (°)")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

# Summary Table
df_events_summary = pd.DataFrame(event_results)[["trial", "posture", "t_apex", "t_ic", "t_km", "duration", "peak_knee", "lumbar_km"]]
df_events_summary.columns = ["Trial", "Posture", "Apex (s)", "IC (s)", "KM (s)", "Absorption Dur (s)", "Peak Knee (°)", "Lumbar Angle (°)"]
print("\n=== Kinematic Events Summary ===")
display(df_events_summary)


---
## Part 3: Step 2 - Inverse Kinetics & Vertical Ground Reaction Forces ($vGRF$)
During a jump landing, the body absorbs kinetic energy from deceleration:
$$\sum F_y = m \cdot a_{COM,y} \implies F_{vGRF} = m(g + a_{COM,y})$$
- **Body Weight Normalization:** Normalizing forces to Body Weight ($BW = m \cdot 9.81\text{ N}$) allows meaningful comparison across participants of different body sizes.
- **Vertical Impulse:** $J_y = \int_{t_{IC}}^{t_{KM}} vGRF(t)\, dt$ represents the total momentum dissipated during landing absorption.
- **Loading Rate:** $\frac{\Delta vGRF}{\Delta t}$ represents how rapidly impact forces rise upon ground contact.


In [ ]:
# Load Dynamics (GRF and Muscle Forces)
def load_trial_dynamics(session_dir, trial_name):
    dyn_dir = os.path.join(session_dir, "OpenSimData", "Dynamics", trial_name)
    if not os.path.isdir(dyn_dir):
        raise FileNotFoundError(f"Dynamics folder not found: {dyn_dir}")
    
    # 1. GRF
    grf_file = None
    for cand in [f"GRF_resultant_{trial_name}_0.mot", f"GRF_{trial_name}_0.mot", "GRF.mot", "GRF.sto"]:
        p = os.path.join(dyn_dir, cand)
        if os.path.isfile(p):
            grf_file = p
            break
    df_grf = read_storage(grf_file)

    # 2. Muscle Forces
    forces_file = None
    for cand in [f"forces_{trial_name}_0.mot", "muscle_forces.sto", "forces.mot"]:
        p = os.path.join(dyn_dir, cand)
        if os.path.isfile(p):
            forces_file = p
            break
    df_forces = read_storage(forces_file)

    return df_grf, df_forces

# Test loading for all trials
trials = df_events_summary["Trial"].tolist()
dynamics_data = {}
for tr in trials:
    df_g, df_f = load_trial_dynamics(session_dir, tr)
    dynamics_data[tr] = {"grf": df_g, "forces": df_f}
    print(f"✓ Loaded dynamics for {tr}: {len(df_g)} timepoints.")


---
## Part 4: Knee Extensor Muscle Forces Analysis
The **Quadriceps Femoris** is the primary muscle group resisting knee flexion during landing absorption:
1. **`vaslat` (Vastus Lateralis):** The largest vastus muscle; contributes the greatest proportion of knee extensor torque.
2. **`vasmed` (Vastus Medialis):** Stabilizes patellar tracking medially.
3. **`vasint` (Vastus Intermedius):** Deep uniarticular extensor.
4. **`recfem` (Rectus Femoris):** **Biarticular** muscle crossing both the hip and knee. It acts simultaneously as a **hip flexor** and **knee extensor**.
   - *Biomechanical Note:* In an upright landing (extended hip), the rectus femoris is lengthened across the hip, altering its active force production compared to a forward-leaning posture (flexed hip).


In [ ]:
# Plot Individual Trial Landing Dynamics
BW_N = SUBJECT_MASS_KG * 9.80665

for tr in trials:
    df_g = dynamics_data[tr]["grf"]
    df_f = dynamics_data[tr]["forces"]
    ev = next(e for e in event_results if e["trial"] == tr)
    
    t_dyn = df_g["time"].values
    mask = (t_dyn >= ev["t_ic"] - 0.05) & (t_dyn <= ev["t_km"] + 0.1)
    t_sub = t_dyn[mask]
    
    total_vgrf = df_g["ground_force_right_vy"].values[mask] + df_g["ground_force_left_vy"].values[mask]
    vgrf_bw = total_vgrf / BW_N

    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Landing Dynamics: {tr} ({ev['posture']})", fontsize=13, fontweight="bold")

    # (A) vGRF
    axs[0].plot(t_sub, vgrf_bw, color="crimson", lw=2.5, label="Total vGRF (BW)")
    axs[0].axvline(ev["t_ic"], color="red", linestyle="--", label="Initial Contact")
    axs[0].axvline(ev["t_km"], color="darkorange", linestyle="-.", label="Knee Max")
    axs[0].axhline(1.0, color="gray", linestyle=":", label="1 Body Weight")
    axs[0].set_title("Vertical Ground Reaction Force", fontweight="bold")
    axs[0].set_xlabel("Time (s)")
    axs[0].set_ylabel("vGRF (BW)")
    axs[0].grid(True, linestyle="--", alpha=0.5)
    axs[0].legend(loc="upper right")

    # (B) Knee Extensor Muscle Forces
    axs[1].plot(t_sub, df_f["vaslat_r"].values[mask], lw=2, label="Vastus Lateralis (vaslat_r)")
    axs[1].plot(t_sub, df_f["vasmed_r"].values[mask], lw=2, label="Vastus Medialis (vasmed_r)")
    axs[1].plot(t_sub, df_f["vasint_r"].values[mask], lw=2, label="Vastus Intermedius (vasint_r)")
    axs[1].plot(t_sub, df_f["recfem_r"].values[mask], lw=2.5, color="magenta", label="Rectus Femoris (recfem_r)")
    axs[1].axvline(ev["t_ic"], color="red", linestyle="--")
    axs[1].axvline(ev["t_km"], color="darkorange", linestyle="-.")
    axs[1].set_title("Right Knee Extensor Muscle Forces", fontweight="bold")
    axs[1].set_xlabel("Time (s)")
    axs[1].set_ylabel("Force (N)")
    axs[1].grid(True, linestyle="--", alpha=0.5)
    axs[1].legend(loc="upper right")

    plt.tight_layout()
    plt.show()


---
## Part 5: Posture Comparison (Forward Lean vs. Upright Landing)
To directly compare across trials with different landing durations, we normalize the absorption phase from **0% (Initial Contact)** to **100% (Maximum Knee Flexion)** using 101-point linear interpolation.
We then compute:
- **Peak Total vGRF ($BW$ and $N$)**
- **Vertical Impulse ($N\cdot s$ and $BW\cdot s$)**
- **Loading Rate ($BW/s$)**
- **Peak Muscle Forces for `recfem`, `vasint`, `vaslat`, `vasmed`, and Total Quads**


In [ ]:
# Time-Normalization and Posture Comparison
def time_normalize(time_arr, data_arr, t_start, t_end, num_points=101):
    mask = (time_arr >= t_start) & (time_arr <= t_end)
    t_crop = time_arr[mask]
    d_crop = data_arr[mask]
    if len(t_crop) < 2:
        idx_start = max(0, np.searchsorted(time_arr, t_start) - 1)
        idx_end = min(len(time_arr), np.searchsorted(time_arr, t_end) + 1)
        t_crop = time_arr[idx_start:idx_end]
        d_crop = data_arr[idx_start:idx_end]
    f = interp1d(t_crop, d_crop, kind="linear", fill_value="extrapolate")
    return f(np.linspace(t_start, t_end, num_points))

pct_grid = np.linspace(0, 100, 101)
trial_curves = {}
comparison_summary = []

trapz_func = getattr(np, 'trapezoid', getattr(np, 'trapz', None))

for tr in trials:
    ev = next(e for e in event_results if e["trial"] == tr)
    df_g = dynamics_data[tr]["grf"]
    df_f = dynamics_data[tr]["forces"]
    df_k = read_storage(os.path.join(session_dir, "OpenSimData", "Kinematics", f"{tr}.mot"))
    
    t_dyn = df_g["time"].values
    t_kin = df_k["time"].values
    
    # Extract total vGRF
    tot_vgrf = df_g["ground_force_right_vy"].values + df_g["ground_force_left_vy"].values
    knee_col = "knee_angle_r" if "knee_angle_r" in df_k.columns else "knee_flexion_r"
    knee_angle = np.abs(df_k[knee_col].values)
    
    norm_vgrf = time_normalize(t_dyn, tot_vgrf, ev["t_ic"], ev["t_km"])
    norm_knee = time_normalize(t_kin, knee_angle, ev["t_ic"], ev["t_km"])
    norm_vaslat = time_normalize(t_dyn, df_f["vaslat_r"].values, ev["t_ic"], ev["t_km"])
    norm_vasmed = time_normalize(t_dyn, df_f["vasmed_r"].values, ev["t_ic"], ev["t_km"])
    norm_vasint = time_normalize(t_dyn, df_f["vasint_r"].values, ev["t_ic"], ev["t_km"])
    norm_recfem = time_normalize(t_dyn, df_f["recfem_r"].values, ev["t_ic"], ev["t_km"])
    norm_quads = norm_vaslat + norm_vasmed + norm_vasint + norm_recfem

    # Impulse & Loading Rate
    dur = ev["duration"]
    peak_vgrf_bw = float(np.max(norm_vgrf) / BW_N)
    impulse_ns = float(trapz_func(norm_vgrf, np.linspace(ev["t_ic"], ev["t_km"], 101)))
    impulse_bws = impulse_ns / BW_N
    
    idx_pk = np.argmax(norm_vgrf)
    t_to_peak = max(0.01, (idx_pk / 100.0) * dur)
    loading_rate = (peak_vgrf_bw - (norm_vgrf[0] / BW_N)) / t_to_peak

    trial_curves[tr] = {
        "vgrf_bw": norm_vgrf / BW_N,
        "vgrf_n": norm_vgrf,
        "knee_deg": norm_knee,
        "vaslat": norm_vaslat,
        "vasmed": norm_vasmed,
        "vasint": norm_vasint,
        "recfem": norm_recfem,
        "quads": norm_quads
    }

    comparison_summary.append({
        "Trial": tr,
        "Posture": ev["posture"],
        "Absorption Duration (s)": round(dur, 3),
        "Peak Knee Flexion (°)": round(ev["peak_knee"], 1),
        "Trunk/Lumbar (°)": round(ev["lumbar_km"], 1),
        "Peak vGRF (BW)": round(peak_vgrf_bw, 2),
        "Peak vGRF (N)": round(float(np.max(norm_vgrf)), 1),
        "Vertical Impulse (BW·s)": round(impulse_bws, 2),
        "Loading Rate (BW/s)": round(loading_rate, 1),
        "vaslat_r Peak (N)": round(float(np.max(norm_vaslat)), 1),
        "vasmed_r Peak (N)": round(float(np.max(norm_vasmed)), 1),
        "vasint_r Peak (N)": round(float(np.max(norm_vasint)), 1),
        "recfem_r Peak (N)": round(float(np.max(norm_recfem)), 1),
        "Total Quads Peak (N)": round(float(np.max(norm_quads)), 1)
    })

df_comp = pd.DataFrame(comparison_summary)
print("=== Landing Posture Comparison Summary ===")
display(df_comp)


In [ ]:
# 4-Panel Comparison Figure (Overlaid Kinematics, GRF, & Quads)
colors = {"LeaningJump": "#1f77b4", "UprightJump": "#d62728"}
labels = {"LeaningJump": "Forward Lean Landing", "UprightJump": "Upright Landing"}

fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"Biomechanics of Jump Landing: Posture Comparison\nSubject Mass = {SUBJECT_MASS_KG} kg", fontsize=14, fontweight="bold")

# Panel A: vGRF
for tr in trials:
    axs[0, 0].plot(pct_grid, trial_curves[tr]["vgrf_bw"], color=colors.get(tr, "black"), lw=2.5, label=labels.get(tr, tr))
axs[0, 0].axhline(1.0, color="gray", linestyle=":", lw=1.5, label="1.0 BW")
axs[0, 0].set_title("(A) Vertical Ground Reaction Force", fontweight="bold")
axs[0, 0].set_xlabel("% Landing Absorption Phase (IC to KM)")
axs[0, 0].set_ylabel("Total vGRF (Body Weights, BW)")
axs[0, 0].grid(True, linestyle="--", alpha=0.5)
axs[0, 0].legend(loc="upper right")

# Panel B: Knee Flexion
for tr in trials:
    axs[0, 1].plot(pct_grid, trial_curves[tr]["knee_deg"], color=colors.get(tr, "black"), lw=2.5, label=labels.get(tr, tr))
axs[0, 1].set_title("(B) Knee Flexion Kinematics", fontweight="bold")
axs[0, 1].set_xlabel("% Landing Absorption Phase")
axs[0, 1].set_ylabel("Knee Angle (°)")
axs[0, 1].grid(True, linestyle="--", alpha=0.5)
axs[0, 1].legend(loc="upper left")

# Panel C: Total Quadriceps Force
for tr in trials:
    axs[1, 0].plot(pct_grid, trial_curves[tr]["quads"], color=colors.get(tr, "black"), lw=2.5, label=labels.get(tr, tr))
axs[1, 0].set_title("(C) Total Quadriceps Force (Right Leg)", fontweight="bold")
axs[1, 0].set_xlabel("% Landing Absorption Phase")
axs[1, 0].set_ylabel("Total Force (N)")
axs[1, 0].grid(True, linestyle="--", alpha=0.5)
axs[1, 0].legend(loc="upper right")

# Panel D: Bar Comparison of Peak Demands
x_idx = np.arange(len(trials))
bar_w = 0.35
pk_vgrf = [row["Peak vGRF (BW)"] for row in comparison_summary]
pk_quads_bw = [row["Total Quads Peak (N)"] / BW_N for row in comparison_summary]

axs[1, 1].bar(x_idx - bar_w/2, pk_vgrf, width=bar_w, label="Peak vGRF (BW)", color="#ff7f0e", alpha=0.85)
axs[1, 1].bar(x_idx + bar_w/2, pk_quads_bw, width=bar_w, label="Peak Quads (BW)", color="#2ca02c", alpha=0.85)
axs[1, 1].set_title("(D) Peak Loading Demand Comparison", fontweight="bold")
axs[1, 1].set_xticks(x_idx)
axs[1, 1].set_xticklabels([labels.get(tr, tr) for tr in trials], fontweight="bold")
axs[1, 1].set_ylabel("Normalized Force (BW)")
axs[1, 1].grid(True, linestyle="--", alpha=0.5, axis="y")
axs[1, 1].legend(loc="upper left")

for i, (v, q) in enumerate(zip(pk_vgrf, pk_quads_bw)):
    axs[1, 1].text(i - bar_w/2, v + 0.1, f"{v:.2f} BW", ha="center", fontsize=9, fontweight="bold")
    axs[1, 1].text(i + bar_w/2, q + 0.15, f"{q:.2f} BW", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig("JumpLanding_Posture_Comparison_Overlay.png", dpi=200)
plt.show()


In [ ]:
# Detailed 4-Knee-Extensor Comparison (recfem, vasint, vaslat, vasmed)
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Knee Extensor Muscle Forces Comparison (Forward Lean vs. Upright)", fontsize=14, fontweight="bold")

muscles = [
    ("vaslat", "Vastus Lateralis (vaslat_r)", axs[0, 0]),
    ("vasmed", "Vastus Medialis (vasmed_r)", axs[0, 1]),
    ("vasint", "Vastus Intermedius (vasint_r)", axs[1, 0]),
    ("recfem", "Rectus Femoris (recfem_r, Biarticular)", axs[1, 1])
]

for key, title, ax in muscles:
    for tr in trials:
        ax.plot(pct_grid, trial_curves[tr][key], color=colors.get(tr, "black"), lw=2.5, label=labels.get(tr, tr))
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("% Landing Phase")
    ax.set_ylabel("Muscle Force (N)")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig("JumpLanding_KneeExtensors_Detailed.png", dpi=200)
plt.show()


---
## Part 6: Export Results & Lab Report Reflection Questions

### Excel Summary Workbook Export
Run the cell below to export the compiled metrics and 101-point curves to `JumpLanding_Posture_Comparison.xlsx`.


In [ ]:
# Export Multi-Sheet Excel Workbook
curves_dict = {"Percent_Landing_Phase": pct_grid}
for tr in trials:
    pfx = f"{tr}_"
    curves_dict[f"{pfx}vGRF_BW"] = trial_curves[tr]["vgrf_bw"]
    curves_dict[f"{pfx}vGRF_N"] = trial_curves[tr]["vgrf_n"]
    curves_dict[f"{pfx}KneeFlexion_deg"] = trial_curves[tr]["knee_deg"]
    curves_dict[f"{pfx}recfem_N"] = trial_curves[tr]["recfem"]
    curves_dict[f"{pfx}vaslat_N"] = trial_curves[tr]["vaslat"]
    curves_dict[f"{pfx}vasmed_N"] = trial_curves[tr]["vasmed"]
    curves_dict[f"{pfx}vasint_N"] = trial_curves[tr]["vasint"]
    curves_dict[f"{pfx}TotalQuads_N"] = trial_curves[tr]["quads"]
df_curves_export = pd.DataFrame(curves_dict)

excel_path = "JumpLanding_Posture_Comparison.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_comp.to_excel(writer, sheet_name="Summary_Metrics", index=False)
    df_curves_export.to_excel(writer, sheet_name="Normalized_Curves", index=False)

print(f"✓ Successfully exported summary workbook to: {excel_path}")

# In Google Colab, trigger download
try:
    from google.colab import files
    files.download(excel_path)
    files.download("JumpLanding_Posture_Comparison_Overlay.png")
    files.download("JumpLanding_KneeExtensors_Detailed.png")
    print("✓ Triggered browser download of report figures and Excel spreadsheet!")
except ImportError:
    print(f"Files saved locally in current working directory: {os.getcwd()}")


---
### Lab Report Discussion & Reflection Questions
Review your generated plots and summary tables to answer the following questions:

#### **Question 1: Impact Attenuation & Impulse-Momentum Principles**
- Compare the **absorption duration** and **peak vGRF** between Upright and Forward Lean landings.
- Based on the impulse-momentum relationship ($J = \Delta p = \int F\, dt = m\Delta v$), why does landing with greater knee and hip compliance reduce peak impact forces and loading rate?

*Student Answer:*  
*(Type your response here)*

---

#### **Question 2: Biarticular Rectus Femoris Mechanics**
- How does the peak force of **`recfem`** differ between Upright and Forward Lean postures?
- Given that `recfem` crosses both the hip (as a flexor) and knee (as an extensor), explain how trunk posture (hip flexion angle) affects the operating length of `recfem` on its force-length curve during landing.

*Student Answer:*  
*(Type your response here)*

---

#### **Question 3: Vasti Force Distribution**
- Among the vasti muscles (`vaslat`, `vasmed`, `vasint`), which muscle generated the highest absolute force?
- What anatomical factor explains this dominant contribution?

*Student Answer:*  
*(Type your response here)*

---

#### **Question 4: ACL Injury Risk Implications**
- Biomechanical studies have established that landing with shallow knee flexion angles combined with high quadriceps contraction forces creates significant anterior shear force on the proximal tibia, straining the Anterior Cruciate Ligament (ACL).
- Which landing posture exhibited higher peak quadriceps demand and stiffer landing kinematics? What cues would you provide to an athlete to mitigate ACL injury risk during jump landings?

*Student Answer:*  
*(Type your response here)*

---

#### **Question 5: Energy Dissipation Strategy (Hip vs. Knee)**
- In forward-lean landing, the total vertical impulse is sustained, but knee extensor forces are reduced. Which muscle groups in the posterior chain (e.g., gluteus maximus, hamstrings) are recruited to absorb the remaining mechanical energy?

*Student Answer:*  
*(Type your response here)*
